In [6]:
%pip uninstall latentmi -y
%pip install git+https://github.com/0kutsu/latentmi

Found existing installation: latentmi 0.1.3
Uninstalling latentmi-0.1.3:
  Successfully uninstalled latentmi-0.1.3
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/0kutsu/latentmi to /tmp/pip-req-build-v_npl57p
  Running command git clone --filter=blob:none --quiet https://github.com/0kutsu/latentmi /tmp/pip-req-build-v_npl57p
  Resolved https://github.com/0kutsu/latentmi to commit a95ec552314fdc4d786a2323f69284fb4425e006
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for latentmi: filename=latentmi-0.1.3-py3-none-any.whl size=9991 sha256=fbb52d6ccd4cfb6dde85cceb910a876b2d719f8ff088f2afbe04c022f41b3890
  Stored in directory: /tmp/pip-ephem-wheel-cache-5hlhuj99/wheels/c1/bd/07/953b6a903fd435834eb94f09dad6fed0cee2fcb3cb879a26d6
Successfully built latentmi
Note: you may need to restart the kernel to use updated packages.


In [1]:
from zoneinfo import ZoneInfo
from datetime import datetime

# CURRENT TIME METHOD
def current_time():
    # Gets the current local date and time
    now = datetime.now(ZoneInfo("America/New_York"))
    
    # Formats and prints just the time (Hour:Minute:Second)
    current_time = now.strftime("%H:%M:%S")
    return current_time

In [5]:
# TESTING THE FUNCTION
from latentmi import lmi
import torch
import numpy as np
import gc

torch.manual_seed(1)
np.random.seed(1)

truth_mi = -0.5*np.log2((1-(4/(np.sqrt(6*3.5)))**2)) # 1.035 bits
trials = 100

print("Time,Trial,LMI Estimate,Variance Estimate,Variance Error")

lmi_estimates = np.empty(trials)
stdvar_estimates = np.empty(trials)
for i in range(trials):
    print(f"{current_time()},{i + 1},", end = "")

    intrinsic = np.random.multivariate_normal(mean=[0, 0], cov=[[6, 4], [4, 3.5]], size=1000)

    # reparameterizing 
    X_intrinsic = intrinsic[:, [0]]
    Y_intrinsic = intrinsic[:, [1]]
                                            
    X_proj = np.random.normal(size=(1, 100))
    Y_proj = np.random.normal(size=(1, 100))

    Xs = X_intrinsic @ X_proj
    Ys = Y_intrinsic @ Y_proj
    
    # testing k = 3, N_dims = 8.
    # LMI estimation
    pmis, _, _ = lmi.estimate(Xs, Ys, regularizer='models.AECross', alpha=1, lam=1, N_dims=8, k=3, 
                              validation_split=0.5, estimate_on_val=True, batch_size=512, lr=0.0001, 
                              epochs=300, patience=30, quiet=True, device=None)
    lmi_estimates[i] = np.nanmean(pmis)
    # variance estimation
    var_estimate, var_error = lmi.estimate_variance(Xs, Ys, n_partitions=9, regularizer='models.AECross', 
                                                    alpha=1, lam=1, N_dims=8, k=3, validation_split=0.5, 
                                                    estimate_on_val=True, batch_size=512, lr=0.0001, epochs=300, 
                                                    patience=30, quiet=True, device=None)
    stdvar_estimates[i] = np.sqrt(var_estimate)

    print(f"%.6f,%.6f,%.6f" % (lmi_estimates[i], var_estimate, var_error))

    torch.cuda.empty_cache()
    gc.collect()

z_scores = (lmi_estimates - truth_mi) / stdvar_estimates

print("\nMean of z-scores: %.6f" % np.mean(z_scores))
print("Standard deviation of z-scores: %.6f" % np.std(z_scores))

Time,Trial,LMI Estimate,Variance Estimate,Variance Error
11:32:31,1,1.130926,0.004905,0.001034
11:33:13,2,1.135973,0.004516,0.000952
11:33:55,3,1.004124,0.003799,0.000801
11:34:40,4,0.835376,0.004777,0.001007
11:35:21,5,1.057415,0.006225,0.001312
11:36:03,6,0.983233,0.003085,0.000650
11:36:44,7,1.089397,0.004300,0.000907
11:37:27,8,1.029970,0.004573,0.000964
11:38:10,9,1.041594,0.004603,0.000970
11:38:51,10,0.998977,0.007542,0.001590
11:39:32,11,1.137558,0.004857,0.001024
11:40:15,12,0.977735,0.004977,0.001049
11:40:57,13,0.995022,0.005696,0.001201
11:41:39,14,1.055653,0.005530,0.001166
11:42:24,15,1.102237,0.003950,0.000833
11:43:07,16,0.973914,0.002389,0.000504
11:43:48,17,1.046092,0.002581,0.000544
11:44:29,18,0.988683,0.004529,0.000955
11:45:12,19,1.200465,0.004143,0.000873
11:45:56,20,1.035404,0.004693,0.000989
11:46:39,21,0.897330,0.002789,0.000588
11:47:22,22,0.965846,0.004442,0.000936
11:48:04,23,1.125865,0.004192,0.000884
11:48:46,24,1.117474,0.005494,0.001158
11:49:31,25,1.14

In [4]:
# SANITY CHECKING VARIANCE ESTIMATES
from latentmi import lmi
import torch 
import numpy as np
import gc
import matplotlib.pyplot as plt

torch.manual_seed(1)
np.random.seed(1)

sample_sizes = [100, 200, 300, 400, 500, 600, 700, 800, 900,
                1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000]

print("Timestamp,Sample Size,Variance Estimate,Standard Error,LMI Estimate")

variance_estimates = np.empty((len(sample_sizes), 3)) # cols: sample size, variance estimate, variance error
for i in range(len(sample_sizes)):
    sample_size = sample_sizes[i]
    print(f"{current_time()},{sample_size},", end = "")

    intrinsic = np.random.multivariate_normal(mean=[0, 0], cov=[[6, 4], [4, 3.5]], size=sample_size)

    # reparameterizing 
    X_intrinsic = intrinsic[:, [0]]
    Y_intrinsic = intrinsic[:, [1]]
                                            
    X_proj = np.random.normal(size=(1, 100))
    Y_proj = np.random.normal(size=(1, 100))

    Xs = X_intrinsic @ X_proj
    Ys = Y_intrinsic @ Y_proj
    
    # lmi estimation
    pmis, _, _ = lmi.estimate(Xs, Ys, regularizer='models.AECross', alpha=1, lam=1, N_dims=8, k=3, 
                              validation_split=0.5, estimate_on_val=True, batch_size=512, lr=0.0001, 
                              epochs=300, patience=30, quiet=True, device=None)
    lmi_estimate = np.nanmean(pmis)
    # variance estimation
    var_estimate, var_error = lmi.estimate_variance(Xs, Ys, n_partitions=9, regularizer='models.AECross', 
                                                    alpha=1, lam=1, N_dims=8, k=3, validation_split=0.5, 
                                                    estimate_on_val=True, batch_size=512, lr=0.0001, epochs=300, 
                                                    patience=30, quiet=True, device=None)
    variance_estimates[i] = [sample_size, var_estimate, var_error]

    print("%.6f,%.6f,%.6f" % (var_estimate, var_error, lmi_estimate))

    torch.cuda.empty_cache()
    gc.collect()

# creating a scatter plot
plt.errorbar(variance_estimates[:, 0], variance_estimates[:, 1], yerr=variance_estimates[:, 2], fmt='s', ecolor='black', capsize=5)
plt.xlabel('Sample Size')
plt.ylabel('Variance Estimate')
plt.title('Variance Estimate vs Sample Size')

plt.xscale('log')
plt.yscale('log')

plt.show()

Timestamp,Sample Size,Variance Estimate,Standard Error,LMI Estimate
11:32:17,100,

KeyboardInterrupt: 